# LAB 4: Vietnamese Spell-checking Pipeline using PhoBERT

This notebook implements a complete Vietnamese spell-checking system using PhoBERT (Vietnamese BERT) for intelligent error detection and correction.

## 0. Installation and Imports

First, let's install all required dependencies and import necessary libraries.

In [1]:
# Install required packages
%pip install transformers
%pip install torch
%pip install torchvision
%pip install torchaudio
%pip install dict

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of dict to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Discarding https://files.pythonhosted.org/packages/67/fb/6a2458c82f59b4aad53949776608d97a46483c403df1dc20c39b413efe10/dict-2020.7.1.tar.gz (from https://pypi.org/simple/dict/): Requested dict from https://files.pythonhosted.org/packages/67/fb/6a2458c82f59b4aad53949776608d97a46483c403df1dc20c39b413efe10/dict-2020.7.1.tar.gz has inconsistent version: expected '2020.7.1', but metadata has '0.0.0'
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requir

In [2]:
# Import necessary libraries
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
import numpy as np
import os
import re
import difflib
import unicodedata
from typing import List, Tuple

## 1. Model Loading

Load PhoBERT model with comprehensive fallback strategy to ensure compatibility.

In [3]:
os.environ.pop('HUGGINGFACE_HUB_TOKEN', None)
os.environ.pop('HF_TOKEN', None)

try:
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", token=False)
    model = AutoModelForMaskedLM.from_pretrained("vinai/phobert-base", token=False)
    print(f"PhoBERT base loaded successfully!")
except Exception as e:
    print(f"Failed to load PhoBERT base: {e}")


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] RobertaForMaskedLM LOAD REPORT from: vinai/phobert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PhoBERT base loaded successfully!


## 2. Masking Words

Function to create masked versions of sentences for spell-checking analysis.

In [16]:
def mask_words_sequentially(sentence: str) -> List[Tuple[str, str, int]]:
    """
    Masks each word in a sentence sequentially for spell-checking.

    Args:
        sentence (str): Input sentence to process

    Returns:
        List[Tuple[str, str, int]]: List of (masked_sentence, original_word, word_position)
    """
    # Tokenize the sentence (simple word-based tokenization)
    words = sentence.split()
    results = []

    for i, word in enumerate(words):
        # TODO: Implement masking logic
        masked_words = words.copy()
        masked_words[i] = tokenizer.mask_token
        masked_sentence = " ".join(masked_words)
        results.append((masked_sentence, word, i))
        # END OF TODO
    return results

In [17]:
# Test the function
print("🧪 Testing mask_words_sequentially function:")
test_sentence = "Bánh mì là một loại đồ án của Việt Nam"
masked_results = mask_words_sequentially(test_sentence)

for masked_sentence, original_word, position in masked_results[:3]:  # Show first 3 examples
    print(f"Position {position+1}: {masked_sentence}")
    print(f"Original word: '{original_word}'")
    print("-" * 50)

🧪 Testing mask_words_sequentially function:
Position 1: <mask> mì là một loại đồ án của Việt Nam
Original word: 'Bánh'
--------------------------------------------------
Position 2: Bánh <mask> là một loại đồ án của Việt Nam
Original word: 'mì'
--------------------------------------------------
Position 3: Bánh mì <mask> một loại đồ án của Việt Nam
Original word: 'là'
--------------------------------------------------


## 3. Calculate Word Probability

Function to calculate how likely a word fits in a masked position using PhoBERT.

In [20]:
def calculate_word_probability(masked_sentence: str, original_word: str) -> float:
    """
    Calculate the probability of a word fitting in a masked position using BERT.

    Args:
        masked_sentence (str): Sentence with [MASK] token
        original_word (str): The original word to check probability for

    Returns:
        float: Probability score between 0 and 1
    """
    try:
        # TODO: Implement probability calculation logic for the original_word in the masked_sentence

        # Step 1: Tokenize the original word
        original_token_ids = tokenizer.encode(
            original_word,
            add_special_tokens=False
        )

        if not original_token_ids:
            return 0.0

        # Step 2: Normalize and replace mask tokens
        mask_token = tokenizer.mask_token
        num_tokens = len(original_token_ids)

        # Convert [MASK] to <mask> for PhoBERT
        masked_sentence = masked_sentence.replace("[MASK]", mask_token)

        # Validate the number of masks
        if masked_sentence.count(mask_token) != 1:
            raise ValueError("Sentence must contain exactly one mask token")

        # Replace with the correct number of masks
        masked_sentence = masked_sentence.replace(
            mask_token,
            " ".join([mask_token] * num_tokens),
            1
        )

        # Step 3: Tokenize the masked sentence
        inputs = tokenizer(masked_sentence, return_tensors="pt")
        inputs = inputs.to(next(model.parameters()).device)

        # Step 4: Find the mask positions
        mask_positions = (
            inputs["input_ids"][0] == tokenizer.mask_token_id
        ).nonzero(as_tuple=True)[0]

        if len(mask_positions) != num_tokens:
            raise ValueError(
                f"Expected {num_tokens} mask positions, "
                f"but found {len(mask_positions)}"
            )

        # Step 5: Get model predictions
        model.eval()

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

        # Step 6: Calculate token probabilities
        probabilities = torch.softmax(
            logits[0, mask_positions, :],
            dim=-1
        )

        # Step 7: Calculate the original word probability
        probability = 1.0

        for i, token_id in enumerate(original_token_ids):
            probability *= probabilities[i, token_id].item()

        # END OF TODO
        return probability

    except Exception as e:
        print(f"Error calculating probability for '{original_word}': {e}")
        return 0.0

In [21]:
# Test the function
print("🧪 Testing calculate_word_probability function:")
test_masked_sentence = "Bánh mì là một loại đồ [MASK] của Việt Nam"
prob_correct = calculate_word_probability(test_masked_sentence, "ăn")
prob_incorrect = calculate_word_probability(test_masked_sentence, "án")

print(f"Probability for correct word 'ăn': {prob_correct:.6f}")
print(f"Probability for incorrect word 'án': {prob_incorrect:.6f}")
print(f"Difference: {prob_correct - prob_incorrect:.6f}")

🧪 Testing calculate_word_probability function:
Probability for correct word 'ăn': 0.432657
Probability for incorrect word 'án': 0.000001
Difference: 0.432657


## 4. Main Function: Spell Checking

Function to analyze entire sentences and identify potential spelling errors.

In [24]:
def check_spelling_errors(sentence: str, threshold: float = 0.00001) -> List[Tuple[str, float, bool]]:
    """
    Check for spelling errors in a sentence using probability threshold.

    Args:
        sentence (str): Input sentence to check
        threshold (float): Probability threshold below which a word is considered misspelled

    Returns:
        List[Tuple[str, float, bool]]: List of (word, probability, is_error)
    """
    words = sentence.split()
    results = []

    print(f"Analyzing sentence: '{sentence}'")
    print(f"Using threshold: {threshold}")
    print("=" * 60)

    for i, word in enumerate(words):
        # TODO: Create masked sentence
        masked_words = words.copy()
        masked_words[i] = tokenizer.mask_token
        masked_sentence = " ".join(masked_words)

        # TODO: Calculate probability (prob)
        prob = calculate_word_probability(masked_sentence, word)
        # END OF TODO

        is_error = prob < threshold

        results.append((word, prob, is_error))
        status = "❌ POTENTIAL ERROR" if is_error else "✅ OK"
        confidence = "HIGH" if prob > 0.01 else "MEDIUM" if prob > 0.001 else "LOW"
        print(f"Word: '{word:12}' | Prob: {prob:.6f} | Confidence: {confidence:6} | {status}")

    print("=" * 60)
    errors_found = sum(1 for _, _, is_error in results if is_error)
    total_words = len(words)
    print(f"📊 Summary: {errors_found}/{total_words} words flagged as potential errors ({errors_found/total_words*100:.1f}%)")
    return results

In [25]:
# Test the function
print("🧪 Testing check_spelling_errors function:")
print()

# Test with correct sentence
correct_sentence = "Bánh mì là một loại đồ ăn của Việt Nam"
print("📝 Test 1 - Correct sentence:")
results1 = check_spelling_errors(correct_sentence, threshold=0.001)

🧪 Testing check_spelling_errors function:

📝 Test 1 - Correct sentence:
Analyzing sentence: 'Bánh mì là một loại đồ ăn của Việt Nam'
Using threshold: 0.001
Word: 'Bánh        ' | Prob: 0.072655 | Confidence: HIGH   | ✅ OK
Word: 'mì          ' | Prob: 0.000433 | Confidence: LOW    | ❌ POTENTIAL ERROR
Word: 'là          ' | Prob: 0.989846 | Confidence: HIGH   | ✅ OK
Word: 'một         ' | Prob: 0.995942 | Confidence: HIGH   | ✅ OK
Word: 'loại        ' | Prob: 0.526085 | Confidence: HIGH   | ✅ OK
Word: 'đồ          ' | Prob: 0.004810 | Confidence: MEDIUM | ✅ OK
Word: 'ăn          ' | Prob: 0.432657 | Confidence: HIGH   | ✅ OK
Word: 'của         ' | Prob: 0.010285 | Confidence: HIGH   | ✅ OK
Word: 'Việt        ' | Prob: 0.000807 | Confidence: LOW    | ❌ POTENTIAL ERROR
Word: 'Nam         ' | Prob: 0.000067 | Confidence: LOW    | ❌ POTENTIAL ERROR
📊 Summary: 3/10 words flagged as potential errors (30.0%)


## 5. Prediction Function: Top Suggestions

Function to get top word predictions for masked positions.

In [26]:
def get_top_predictions(masked_sentence: str, top_k: int = 5) -> List[Tuple[str, float]]:
    """
    Get top predictions for a masked position.

    Args:
        masked_sentence (str): Sentence with [MASK] token
        top_k (int): Number of top predictions to return

    Returns:
        List[Tuple[str, float]]: List of (predicted_word, probability)
    """
    try:
        # TODO: Replace [MASK] with the actual mask token
        masked_sentence = masked_sentence.replace("[MASK]", tokenizer.mask_token)

        # TODO: Tokenize and get predictions
        inputs = tokenizer(masked_sentence, return_tensors="pt")
        inputs = inputs.to(next(model.parameters()).device)

        # Find the mask position
        mask_positions = (
            inputs["input_ids"][0] == tokenizer.mask_token_id
        ).nonzero(as_tuple=True)[0]

        if len(mask_positions) != 1:
            raise ValueError("Sentence must contain exactly one mask token")

        model.eval()

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

        # Get probabilities at the masked position
        mask_logits = logits[0, mask_positions.item(), :]
        probabilities = torch.softmax(mask_logits, dim=-1)

        # TODO: Get top k predictions
        top_probs, top_indices = torch.topk(
            probabilities,
            k=min(top_k, probabilities.size(-1))
        )

        results = []

        for token_id, prob in zip(top_indices.tolist(), top_probs.tolist()):
            predicted_word = tokenizer.decode(
                [token_id],
                skip_special_tokens=True
            ).strip()

            results.append((predicted_word, prob))

        # END OF TODO
        return results

    except Exception as e:
        print(f"❌ Error getting predictions: {e}")
        return []

In [27]:
# Test the function
print("🧪 Testing get_top_predictions function:")
test_masked = "Bánh mì là một loại đồ [MASK] của Việt Nam"
predictions = get_top_predictions(test_masked, top_k=5)

print(f"For sentence: '{test_masked}'")
print("Top predictions:")
for i, (word, prob) in enumerate(predictions, 1):
    print(f"  {i}. '{word}' (probability: {prob:.6f})")

🧪 Testing get_top_predictions function:
For sentence: 'Bánh mì là một loại đồ [MASK] của Việt Nam'
Top predictions:
  1. 'ăn' (probability: 0.432657)
  2. 'tráng_miệng' (probability: 0.123274)
  3. 'uống' (probability: 0.087556)
  4. 'ngọt' (probability: 0.053297)
  5. 'ẩm_thực' (probability: 0.027972)


## 6. Find similar words

Helper function for similarity comparison by removing Vietnamese diacritical marks.

### 6.1. Utl: Remove accents

In [28]:
def remove_accents(text: str) -> str:
    """
    Remove Vietnamese accents for similarity comparison.

    Args:
        text (str): Input text with Vietnamese accents

    Returns:
        str: Text without accents
    """
    return ''.join(c for c in unicodedata.normalize('NFD', text)
                   if unicodedata.category(c) != 'Mn')

# Test the function
print("🧪 Testing remove_accents function:")
test_words = ["ăn", "án", "Việt", "việt", "phở", "pho"]

for word in test_words:
    no_accent = remove_accents(word)
    print(f"'{word}' -> '{no_accent}'")

🧪 Testing remove_accents function:
'ăn' -> 'an'
'án' -> 'an'
'Việt' -> 'Viet'
'việt' -> 'viet'
'phở' -> 'pho'
'pho' -> 'pho'


## 6.2. Find Similar Words

In [29]:
def find_similar_words(target_word: str, predictions: List[str], threshold: float = 0.6) -> List[Tuple[str, float]]:
    """
    Find similar words based on string similarity.

    Args:
        target_word (str): The word to find similarities for
        predictions (List[str]): List of candidate words
        threshold (float): Minimum similarity threshold

    Returns:
        List[Tuple[str, float]]: List of (word, similarity_score) pairs
    """
    similar_words = []
    target_clean = remove_accents(target_word.lower())

    for pred in predictions:
        pred_clean = remove_accents(pred.lower())
        similarity = difflib.SequenceMatcher(None, target_clean, pred_clean).ratio()

        if similarity >= threshold:
            similar_words.append((pred, similarity))

    return sorted(similar_words, key=lambda x: x[1], reverse=True)

In [30]:
# Test with example
print("🧪 Testing find_similar_words function:")
target = "việt"
candidates = ["Việt", "viêt", "viet", "vietnam", "anh", "việt"]
results = find_similar_words(target, candidates, threshold=0.5)

print(f"Target word: '{target}'")
print(f"Candidates: {candidates}")
print(f"Similar words (threshold=0.5):")
for word, score in results:
    print(f"  '{word}' -> similarity: {score:.3f}")

🧪 Testing find_similar_words function:
Target word: 'việt'
Candidates: ['Việt', 'viêt', 'viet', 'vietnam', 'anh', 'việt']
Similar words (threshold=0.5):
  'Việt' -> similarity: 1.000
  'viêt' -> similarity: 1.000
  'viet' -> similarity: 1.000
  'việt' -> similarity: 1.000
  'vietnam' -> similarity: 0.727


## 7. Main Function:

In [33]:
from typing import Dict, Any

# Try with the sentence containing errors
TEST_SENTENCES = "Tôi thích ăn phở vò buổi sáng"

def suggest_spelling_corrections(sentence: str, threshold: float = 1.5) -> Dict[str, Any]:
    """
    Main function to detect spelling errors and suggest corrections.

    Args:
        sentence (str): Input sentence to check
        threshold (float): Probability threshold for error detection

    Returns:
        Dict[str, Any]: Complete analysis results
    """
    print(f"🔍 Analyzing sentence: '{sentence}'")
    print("-" * 60)

    # Step 1: Check for spelling errors
    error_results = check_spelling_errors(sentence, threshold=threshold)

    # Filter only the errors (is_error = True)
    errors = [(word, prob, i) for i, (word, prob, is_error) in enumerate(error_results) if is_error]

    if not errors:
        print("No spelling errors detected!")
        return {
            'sentence': sentence,
            'errors_found': False,
            'error_details': [],
            'suggestions': {}
        }

    print(f"Found {len(errors)} potential spelling error(s):")

    suggestions = {}
    error_details = []

    # Step 2: Generate suggestions for each error. Uncomment if you have implemented this part.

    for word, prob_score, position in errors:
        print(f"\nWord '{word}' at position {position} (score: {prob_score:.3f}):")

        # Get masked sentence
        words = sentence.split()
        masked_sentence = ' '.join(words[:position] + ['[MASK]'] + words[position+1:])

        # Get top predictions
        predictions = get_top_predictions(masked_sentence, top_k=10)

        # Find similar words
        pred_words = [pred[0] for pred in predictions]  # Extract words from (word, probability) tuples
        similar_words = find_similar_words(word, pred_words, threshold=0.4)

        print(f"  🔄 Top suggestions:")
        suggestion_list = []
        for i, (suggestion, similarity) in enumerate(similar_words[:5]):
            pred_prob = next((p[1] for p in predictions if p[0] == suggestion), 0.0)
            print(f"    {i+1}. '{suggestion}' (similarity: {similarity:.3f}, fit: {pred_prob:.3f})")
            suggestion_list.append({
                'word': suggestion,
                'similarity': similarity,
                'probability': pred_prob,
                'confidence': (similarity + pred_prob) / 2
            })

        suggestions[word] = suggestion_list
        error_details.append({
            'word': word,
            'position': position,
            'probability_score': prob_score,
            'suggestions': suggestion_list
        })

    return {
        'sentence': sentence,
        'errors_found': True,
        'error_details': error_details,
        'suggestions': suggestions
    }


# Example usage
test_sentence = TEST_SENTENCES
results = suggest_spelling_corrections(test_sentence, threshold=0.00001)

🔍 Analyzing sentence: 'Tôi thích ăn phở vò buổi sáng'
------------------------------------------------------------
Analyzing sentence: 'Tôi thích ăn phở vò buổi sáng'
Using threshold: 1e-05
Word: 'Tôi         ' | Prob: 0.048751 | Confidence: HIGH   | ✅ OK
Word: 'thích       ' | Prob: 0.783187 | Confidence: HIGH   | ✅ OK
Word: 'ăn          ' | Prob: 0.932253 | Confidence: HIGH   | ✅ OK
Word: 'phở         ' | Prob: 0.000099 | Confidence: LOW    | ✅ OK
Word: 'vò          ' | Prob: 0.000006 | Confidence: LOW    | ❌ POTENTIAL ERROR
Word: 'buổi        ' | Prob: 0.647270 | Confidence: HIGH   | ✅ OK
Word: 'sáng        ' | Prob: 0.622676 | Confidence: HIGH   | ✅ OK
📊 Summary: 1/7 words flagged as potential errors (14.3%)
Found 1 potential spelling error(s):

Word 'vò' at position 4 (score: 0.000):
  🔄 Top suggestions:
    1. 'vào' (similarity: 0.800, fit: 0.782)
    2. 'bò' (similarity: 0.500, fit: 0.063)
    3. 'mỗi' (similarity: 0.400, fit: 0.006)
    4. 'xào' (similarity: 0.400, fit: 0.005)
